# Fraud Detection — Random Forest with Time-Based Split

**Approach:**
1. Same feature engineering as the PCA model for a fair comparison
2. Time-based train/test split using `step` — prevents future data leaking into training
3. Random Forest with `balanced_subsample` to handle the 0.13% fraud class imbalance
4. Feature importance analysis + threshold tuning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, precision_recall_curve,
    roc_auc_score, f1_score
)
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [2]:
df = pd.read_csv('Financial_Fraud_dataset.csv')
print(f"Full dataset shape: {df.shape}")
print(f"Overall fraud rate: {df['isFraud'].mean()*100:.4f}%")
print(f"\nStep range: {df['step'].min()} — {df['step'].max()}")

Full dataset shape: (6362620, 11)
Overall fraud rate: 0.1291%

Step range: 1 — 743


## 2. Feature Engineering

Identical to the PCA notebook so results are comparable.

In [3]:
# Filter to types where fraud occurs
df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
print(f"Filtered shape: {df.shape}")
print(f"Fraud rate after filter: {df['isFraud'].mean()*100:.4f}%")

# Encode transaction type (1 = TRANSFER, 0 = CASH_OUT)
df['type_encoded'] = (df['type'] == 'TRANSFER').astype(int)

# Balance discrepancy features — strongest fraud signals
df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

# Account drain: origin emptied after transaction
df['origDrained'] = ((df['newbalanceOrig'] == 0) & (df['oldbalanceOrg'] > 0)).astype(int)

# Destination balance unchanged despite receiving funds
df['destUnchanged'] = (df['newbalanceDest'] == df['oldbalanceDest']).astype(int)

# Log-transform amount
df['log_amount'] = np.log1p(df['amount'])

FEATURES = [
    'log_amount',
    'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'errorBalanceOrig', 'errorBalanceDest',
    'origDrained', 'destUnchanged',
    'type_encoded'
]

print("\nFeature engineering complete.")

Filtered shape: (2770409, 11)
Fraud rate after filter: 0.2965%

Feature engineering complete.


## 3. Time-Based Train/Test Split

**Why time-based?** A random split would let the model 'see' transactions from later time steps during training, which is unrealistic — in production, you always predict on future data. We use the `step` column as a proxy for time.

80% of steps → train, 20% → test.

In [4]:
split_step = int(df['step'].quantile(0.80))
print(f"Split at step {split_step}")
print(f"Train: steps 1–{split_step}")
print(f"Test:  steps {split_step+1}–{df['step'].max()}")

train_df = df[df['step'] <= split_step]
test_df  = df[df['step'] >  split_step]

X_train = train_df[FEATURES].values
y_train = train_df['isFraud'].values
X_test  = test_df[FEATURES].values
y_test  = test_df['isFraud'].values

print(f"\nTrain: {len(X_train):,} samples | Fraud: {y_train.sum():,} ({y_train.mean()*100:.4f}%)")
print(f"Test:  {len(X_test):,} samples  | Fraud: {y_test.sum():,} ({y_test.mean()*100:.4f}%)")

Split at step 354
Train: steps 1–354
Test:  steps 355–743

Train: 2,217,905 samples | Fraud: 3,955 (0.1783%)
Test:  552,504 samples  | Fraud: 4,258 (0.7707%)


## 4. Train Random Forest

Key hyperparameter choices:
- `class_weight='balanced_subsample'` — each tree's bootstrap sample is reweighted so fraud counts more, without globally oversampling
- `max_depth=20` — deep enough to capture complex patterns, bounded to reduce overfitting
- `n_jobs=-1` — use all CPU cores

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=10,       # prevents tiny leaves that overfit to noise
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=42
)

print("Training Random Forest (this may take a few minutes on 2M+ rows)...")
rf.fit(X_train, y_train)
print("Training complete.")

Training Random Forest (this may take a few minutes on 2M+ rows)...


## 5. Evaluate on Test Set

In [ ]:
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

pr_auc  = average_precision_score(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

print("Classification Report (default 0.5 threshold):")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))
print(f"PR-AUC:  {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
plt.title('Confusion Matrix — Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positives  (caught fraud):   {tp:,}")
print(f"False Negatives (missed fraud):   {fn:,}")
print(f"False Positives (false alarms):   {fp:,}")
print(f"True Negatives  (correct legit):  {tn:,}")

## 7. Precision-Recall Curve & Threshold Tuning

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

# Optimal threshold by F1
f1_scores   = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_idx    = np.argmax(f1_scores)
best_thresh = thresholds[best_idx]

plt.figure(figsize=(9, 5))
plt.plot(recall, precision, color='seagreen', lw=2, label=f'PR-AUC = {pr_auc:.4f}')
plt.scatter(recall[best_idx], precision[best_idx], color='red', zorder=5,
            label=f'Best threshold = {best_thresh:.3f} (F1={f1_scores[best_idx]:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Random Forest')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Optimal threshold: {best_thresh:.4f}")
print(f"Best F1:           {f1_scores[best_idx]:.4f}")

y_pred_opt = (y_proba >= best_thresh).astype(int)
print("\nClassification Report at Optimal Threshold:")
print(classification_report(y_test, y_pred_opt, target_names=['Legitimate', 'Fraud']))

## 8. Feature Importances

Random Forest provides Gini-based importance scores — shows which features the model relied on most. This is a major advantage over the PCA approach: the features remain interpretable.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
colors = ['#d62728' if imp > importances.median() else 'steelblue' for imp in importances]
importances.plot(kind='barh', color=colors)
plt.axvline(x=importances.median(), color='gray', linestyle='--', alpha=0.6, label='Median')
plt.xlabel('Gini Importance Score')
plt.title('Feature Importances — Random Forest')
plt.legend()
plt.tight_layout()
plt.show()

print("Feature importances (sorted):")
print(importances.sort_values(ascending=False).to_string())

## 9. Fraud Probability Distribution

Visualize how well the model separates fraud from legitimate transactions by looking at the predicted probability distributions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
axes[0].hist(y_proba[y_test == 0], bins=50, alpha=0.6, label='Legitimate', color='steelblue', density=True)
axes[0].hist(y_proba[y_test == 1], bins=50, alpha=0.6, label='Fraud',      color='red',       density=True)
axes[0].axvline(x=best_thresh, color='black', linestyle='--', label=f'Optimal threshold ({best_thresh:.3f})')
axes[0].set_xlabel('Predicted Fraud Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Fraud Probability Distribution')
axes[0].legend()

# Zoom in on the 0–0.3 range to see separation near threshold
axes[1].hist(y_proba[y_test == 0], bins=50, alpha=0.6, label='Legitimate', color='steelblue', density=True, range=(0, 0.3))
axes[1].hist(y_proba[y_test == 1], bins=50, alpha=0.6, label='Fraud',      color='red',       density=True, range=(0, 0.3))
axes[1].axvline(x=best_thresh, color='black', linestyle='--', label=f'Optimal threshold ({best_thresh:.3f})')
axes[1].set_xlabel('Predicted Fraud Probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Zoomed: 0–0.3 Probability Range')
axes[1].legend()

plt.suptitle('Random Forest — Predicted Probability Distributions', y=1.01)
plt.tight_layout()
plt.show()

## 10. Summary

| Aspect | Random Forest |
|---|---|
| Split strategy | Time-based (no leakage) |
| Imbalance handling | `balanced_subsample` per tree |
| Interpretability | Feature importances |
| Threshold | Tuned for max F1 |
| Key metric | PR-AUC (better than accuracy/ROC for rare events) |

**Comparing to PCA + Logistic Regression:**
- Random Forest captures non-linear interactions (e.g., fraud only when *both* balance error is high *and* account is drained)
- Random Forest features stay interpretable; PCA components are abstract linear combinations
- Logistic Regression is faster and has better-calibrated probabilities
- PR-AUC is the fairest comparison metric between the two models